# 04 - Ablasi: FL DENGAN vs TANPA SFM (Paper 4, Fase 2) - BUKTI UTAMA H1

**Tujuan (klaim inti paper):** tunjukkan bahwa **SFM adalah *enabler*** federasi
lintas-skema. Bandingkan FedAvg di ruang **SFM 9-fitur selaras** melawan dua
upaya **naif tanpa SFM** yang biasa dicoba praktisi ketika klien berskema beda:

- **Naif-1 (by-position):** ambil 9 kolom numerik pertama tiap klien dan tumpuk
  berdasar POSISI. Fitur ke-i CIC != fitur ke-i UNSW secara semantik -> FedAvg
  merata-ratakan bobot pada dimensi yang artinya beda antar-klien.
- **Naif-2 (name-intersection):** pakai HANYA kolom yang namanya sama persis di
  kedua skema. Lazimnya irisan ini kosong/sangat kecil (CICFlowMeter vs Argus
  pakai penamaan beda total) -> federasi praktis gagal / fitur terlalu sedikit.

**Hipotesis H1:** `federated-SFM >> federated-naif` (MCC GLOBAL-test). Jika benar,
ini bukti langsung SFM menyelesaikan heterogenitas skema-fitur, bukan sekadar
'menambah FL ke pipeline lama'.

**Konsistensi:** logika MLP/FedAvg SAMA (`../aws/fed_node.py`); yang berubah HANYA
ruang fitur input. z-score per-klien lokal. Output: `fed_out/ablation_sfm.csv`.

> Jalankan setelah nb01 (SFM) + butuh akses data mentah utk varian naif.
> JANGAN mengarang angka.

## 0. Setup + impor logika FL dari fed_node.py (sumber tunggal)

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef, f1_score, balanced_accuracy_score
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=50; LOCAL_EPOCHS=1; LR=1e-3; SEED=42
print('fed_node loaded | LAYERS', fed.LAYERS)

## 1. Kerangka FedAvg generik (dim fitur bebas)

`init_weights` di fed_node mematok input=9 (ruang SFM). Untuk ablasi naif dgn
dimensi D != 9, kita bentuk bobot layer-pertama berukuran D->64 memakai inisialisasi
He yang SAMA; sisanya identik. Fungsi `forward/train_local/fedavg` fed_node bekerja
untuk D berapa pun karena hanya mengandalkan bentuk W. Ini menjaga logika FL identik,
hanya ruang input yang berbeda (variabel yang diuji).

In [ ]:
def init_weights_dim(D, seed=42):
    rng=np.random.RandomState(seed); layers=[D,64,32,1]; W=[]
    for a,b in zip(layers[:-1],layers[1:]):
        W.append(rng.randn(a,b)*np.sqrt(2.0/a)); W.append(np.zeros(b))
    return W
def run_fedavg(client_data, rounds=ROUNDS, epochs=LOCAL_EPOCHS, lr=LR, seed=SEED):
    # client_data: {cid:(Xtr,ytr)} - X sudah di ruang fitur target (BELUM z-score)
    D=next(iter(client_data.values()))[0].shape[1]
    zc={cid: fed.zscore_fit(X) for cid,(X,y) in client_data.items()}
    Xz={cid: fed.zscore_apply(client_data[cid][0], *zc[cid]) for cid in client_data}
    sizes={cid: len(client_data[cid][1]) for cid in client_data}
    Wg=init_weights_dim(D, seed)
    for t in range(rounds):
        cw=[]; cs=[]
        for cid in client_data:
            Wl=fed.train_local(Wg, Xz[cid], client_data[cid][1],
                               epochs=epochs, lr=lr, mu=0.0, W_global=Wg, seed=seed)
            cw.append(Wl); cs.append(sizes[cid])
        Wg=fed.fedavg(cw, cs)
    return Wg, zc
def predict(W, X, mu, sd):
    out,_=fed.forward(W, fed.zscore_apply(X,mu,sd)); return (out>=0.5).astype(int)
def metrics(y,yp):
    return dict(MCC=round(matthews_corrcoef(y,yp),4),
                F1=round(f1_score(y,yp,zero_division=0),4),
                bal_acc=round(balanced_accuracy_score(y,yp),4))
print('kerangka FedAvg generik siap')

## 2. Ruang DENGAN SFM (dari nb01 - 9 fitur selaras)

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p,allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
Xc_tr,yc_tr=load('cic_train.npz'); Xc_te,yc_te=load('cic_test.npz')
Xu_tr,yu_tr=load('unsw_train.npz'); Xu_te,yu_te=load('unsw_test.npz')
ok_sfm=all(v is not None for v in [Xc_tr,Xu_tr,Xc_te,Xu_te])
res=[]
if ok_sfm:
    Wg,zc=run_fedavg({'cic':(Xc_tr,yc_tr),'unsw':(Xu_tr,yu_tr)})
    Xg_te=np.vstack([Xc_te,Xu_te]); yg_te=np.concatenate([yc_te,yu_te])
    mu_g,sd_g=fed.zscore_fit(np.vstack([Xc_tr,Xu_tr]))
    res.append({'variant':'federated-SFM (9 selaras)','dim':9,'eval':'GLOBAL-test',
                **metrics(yg_te, predict(Wg,Xg_te,mu_g,sd_g))})
    res.append({'variant':'federated-SFM (9 selaras)','dim':9,'eval':'CIC-test',
                **metrics(yc_te, predict(Wg,Xc_te,*zc['cic']))})
    res.append({'variant':'federated-SFM (9 selaras)','dim':9,'eval':'UNSW-test',
                **metrics(yu_te, predict(Wg,Xu_te,*zc['unsw']))})
    print('SFM variant selesai')
else:
    print('(partisi SFM belum ada - jalankan nb01)')
print('=== SEL 2 (SFM) SELESAI ===')

## 3. Ruang TANPA SFM - Naif-1 (by-position) & Naif-2 (name-intersection)

Perlu data MENTAH tiap klien (sebelum pemetaan SFM). Dimuat dari sumber asli
(UNSW CSV, CIC pkl) seperti nb01, tetapi TANPA kamus SFM. Dua konstruksi naif:

- **by-position:** 9 kolom numerik pertama (urutan file) tiap klien, tumpuk by index.
- **name-intersection:** kolom yang namanya identik di kedua skema (biasanya ~0).

Jika data mentah tak tersedia di env ini, sel mencatat `[TBD]` dan dilewati
(JANGAN mengarang).

In [ ]:
import pickle
UNSW_TRAIN='../../unswnb-15/data/UNSW_NB15_training-set.csv'
UNSW_TEST ='../../unswnb-15/data/UNSW_NB15_testing-set.csv'
CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
have_raw = os.path.exists(UNSW_TRAIN) and os.path.exists(CIC_PKL)
def numeric_df(df):
    return df.select_dtypes(include=[np.number]).replace([np.inf,-np.inf],np.nan).fillna(0.0)
raw={}
if have_raw:
    utr=pd.read_csv(UNSW_TRAIN); ute=pd.read_csv(UNSW_TEST)
    yu_tr_r=utr['label'].astype(int).values; yu_te_r=ute['label'].astype(int).values
    un_tr=numeric_df(utr.drop(columns=[c for c in ['label','id'] if c in utr.columns]))
    un_te=numeric_df(ute.drop(columns=[c for c in ['label','id'] if c in ute.columns]))
    d=pickle.load(open(CIC_PKL,'rb'))
    cdf=pd.DataFrame(d['X'],columns=list(d['feature_names'])) if not isinstance(d.get('X'),pd.DataFrame) else d['X']
    benign=d.get('label_mapping',{}).get('Benign',0)
    yc_all=(np.asarray(d['y'])!=benign).astype(int)
    from sklearn.model_selection import train_test_split
    cic_num=numeric_df(cdf)
    ci_tr,ci_te,yc_tr_r,yc_te_r=train_test_split(cic_num,yc_all,test_size=0.3,random_state=SEED,stratify=yc_all)
    raw=dict(un_tr=un_tr,un_te=un_te,yu_tr=yu_tr_r,yu_te=yu_te_r,
             ci_tr=ci_tr,ci_te=ci_te,yc_tr=yc_tr_r,yc_te=yc_te_r)
    print('data mentah dimuat | CIC cols',cic_num.shape[1],'| UNSW cols',un_tr.shape[1])
else:
    print('(data mentah tak tersedia di env ini - varian naif -> [TBD], jalankan di SageMaker)')

In [ ]:
def take_first_k(df, k=9):
    return df.iloc[:, :k].to_numpy(float)
if have_raw:
    # --- Naif-1: by-position (9 kolom pertama) ---
    cd={'cic':(take_first_k(raw['ci_tr']), raw['yc_tr']),
        'unsw':(take_first_k(raw['un_tr']), raw['yu_tr'])}
    Wg1,zc1=run_fedavg(cd)
    Xg_te_pos=np.vstack([take_first_k(raw['ci_te']),take_first_k(raw['un_te'])])
    yg_te_r=np.concatenate([raw['yc_te'],raw['yu_te']])
    mu1,sd1=fed.zscore_fit(np.vstack([cd['cic'][0],cd['unsw'][0]]))
    res.append({'variant':'federated-naif-1 (by-position)','dim':9,'eval':'GLOBAL-test',
                **metrics(yg_te_r, predict(Wg1,Xg_te_pos,mu1,sd1))})
    # --- Naif-2: name-intersection ---
    inter=[c for c in raw['ci_tr'].columns if c in set(raw['un_tr'].columns)]
    print('kolom nama-identik lintas-skema:', len(inter), inter[:10])
    if len(inter)>=1:
        cd2={'cic':(raw['ci_tr'][inter].to_numpy(float), raw['yc_tr']),
             'unsw':(raw['un_tr'][inter].to_numpy(float), raw['yu_tr'])}
        Wg2,zc2=run_fedavg(cd2)
        Xg2=np.vstack([raw['ci_te'][inter].to_numpy(float),raw['un_te'][inter].to_numpy(float)])
        mu2,sd2=fed.zscore_fit(np.vstack([cd2['cic'][0],cd2['unsw'][0]]))
        res.append({'variant':f'federated-naif-2 (name-inter, D={len(inter)})','dim':len(inter),
                    'eval':'GLOBAL-test', **metrics(yg_te_r, predict(Wg2,Xg2,mu2,sd2))})
    else:
        res.append({'variant':'federated-naif-2 (name-inter)','dim':0,'eval':'GLOBAL-test',
                    'MCC':None,'F1':None,'bal_acc':None,
                    'note':'irisan nama KOSONG -> federasi mentah mustahil (bukti kualitatif H1)'})
    print('varian naif selesai')
else:
    res.append({'variant':'federated-naif (by-position & name-inter)','dim':None,
                'eval':'GLOBAL-test','MCC':None,'F1':None,'bal_acc':None,'note':'[TBD] butuh data mentah'})
print('=== SEL 3 (naif tanpa SFM) SELESAI ===')

## 4. Rangkuman ablasi + simpan (bukti H1)

In [ ]:
dfres=pd.DataFrame(res)
dfres.to_csv(os.path.join(OUTDIR,'ablation_sfm.csv'),index=False)
import IPython.display as ipd; ipd.display(dfres)
print('\nH1: federated-SFM (GLOBAL-test MCC) harus JAUH di atas kedua varian naif.')
print('Jika name-intersection kosong -> argumen kualitatif: tanpa SFM, FL lintas-skema')
print('bahkan tak dapat dibentuk. Ini bukti terkuat SFM = enabler.')
print('=== SEL 4 (rangkuman ablasi) SELESAI ===')

## 5. Catatan untuk naskah

- Hasil -> **Tabel E** (`documentation_federated.md`): bukti utama **H1**.
- Dua jenis bukti: (i) KUANTITATIF (SFM MCC >> naif-by-position MCC); (ii)
  KUALITATIF (name-intersection kosong -> FL lintas-skema tak terbentuk tanpa SFM).
- Logika FL identik dgn SFM; yang beda HANYA ruang fitur -> perbandingan adil.
- Jujur: by-position bukan hal yang masuk akal dilakukan praktisi; justru itu
  intinya - tanpa jembatan semantik, penggabungan fitur lintas-tool tak bermakna.
- Angka final diisi dari eksekusi SageMaker (data mentah tersedia di sana).